# Week 4 — Convolutional VAE on MNIST

**Course:** Noise → Masterpiece (Build Stable Diffusion from First Principles) — Week 4

Same VAE as Milestone 1 — encoder → reparameterize → decoder, trained on reconstruction + KL — but the
**linear layers become convolutions** so it can model real 28×28 images. The reconstruction term switches to
**binary cross-entropy** because MNIST pixels live in `[0, 1]` (a Bernoulli likelihood).

We keep the **latent dimension at 2** on purpose: it lets us draw the two iconic VAE-on-MNIST pictures —
a 2D map of where each digit lands in latent space, and a decoded grid showing the digits smoothly morph
across that space. (Raising `LATENT_DIM` sharpens reconstructions further but costs us those 2D views.)

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)   # in Colab: Runtime > Change runtime type > T4 GPU for speed
torch.manual_seed(0)

LATENT_DIM = 2   # 2 keeps the latent-map + manifold-grid views; raise (e.g. 16) for crisper reconstructions

## Step 1 — MNIST data

`ToTensor()` scales pixels to `[0, 1]` and gives shape `[1, 28, 28]` — exactly what BCE expects.
The dataset downloads automatically on first run.

In [ ]:
transform = transforms.ToTensor()
train_set = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False)
print("train:", len(train_set), "| test:", len(test_set))

## Step 2 — The Convolutional VAE

The **encoder** uses strided convolutions to shrink the image while growing channels: `28×28×1 → 14×14×32 →
7×7×64`, then two linear heads produce `μ` and `log σ²`. The **decoder** mirrors it with transpose-convolutions
that grow the spatial size back: `7×7×64 → 14×14×32 → 28×28×1`, ending in a sigmoid so outputs are valid pixels
in `[0, 1]`. The reparameterization step is unchanged from Milestone 1.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        # encoder: 28x28x1 -> 14x14x32 -> 7x7x64 -> flatten
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_dec    = nn.Linear(latent_dim, 64 * 7 * 7)
        # decoder: 7x7x64 -> 14x14x32 -> 28x28x1
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 64, 7, 7)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

## Step 3 — The ELBO loss (BCE reconstruction)

Identical structure to before: reconstruction + KL, both summed. The only change from the 2D version is **BCE
instead of MSE**, because the pixels are Bernoulli (in `[0, 1]`) rather than real-valued.

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    bce = F.binary_cross_entropy(recon_x, x, reduction="sum")
    kl  = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kl, bce, kl

## Step 4 — Train

In [ ]:
def train(model, loader, epochs=10, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(epochs):
        model.train(); total = 0.0
        for x, _ in loader:
            x = x.to(device)
            recon, mu, logvar = model(x)
            loss, _, _ = vae_loss(recon, x, mu, logvar)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        avg = total / len(loader.dataset)
        history.append(avg)
        print(f"epoch {ep+1:2d}/{epochs} | loss/img {avg:.2f}")
    return history

model = ConvVAE(LATENT_DIM)
history = train(model, train_loader, epochs=10)

plt.plot(history, marker="o"); plt.xlabel("epoch"); plt.ylabel("loss per image")
plt.title("Training loss"); plt.show()

## Step 5 — Reconstructions (the headline deliverable)

Top row: real test digits. Bottom row: the VAE's reconstruction. Recognizable digits — clearly sharper and more
structured than a linear VAE could manage.

In [ ]:
model.eval()
x, _ = next(iter(test_loader))
x = x[:10].to(device)
with torch.no_grad():
    recon, _, _ = model(x)
x, recon = x.cpu(), recon.cpu()

fig, ax = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    ax[0, i].imshow(x[i, 0], cmap="gray");     ax[0, i].axis("off")
    ax[1, i].imshow(recon[i, 0], cmap="gray"); ax[1, i].axis("off")
ax[0, 0].set_ylabel("original",  rotation=0, labelpad=40); 
ax[1, 0].set_ylabel("recon",     rotation=0, labelpad=40)
plt.suptitle("Original (top) vs reconstruction (bottom)"); plt.show()

## Step 6 — The latent map

Encode the whole test set and plot each digit's `μ`, colored by its true label. Digits of the same class
cluster together, and the clusters sit next to visually similar digits — structure the model discovered with no
labels.

In [ ]:
mus, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        mu, _ = model.encode(x.to(device))
        mus.append(mu.cpu()); labels.append(y)
mus = torch.cat(mus); labels = torch.cat(labels)

plt.figure(figsize=(8, 7))
sc = plt.scatter(mus[:, 0], mus[:, 1], c=labels, cmap="tab10", s=4, alpha=0.6)
plt.colorbar(sc, ticks=range(10), label="digit")
plt.xlabel("z1"); plt.ylabel("z2"); plt.title("Latent map of the test set"); plt.show()

## Step 7 — The latent manifold (the iconic VAE picture)

Walk a grid of `(z1, z2)` values and decode each point. You see every digit emerge and **smoothly morph** into
its neighbors as you move across the latent plane — direct visual proof that the KL term made the space
continuous and sampleable.

In [ ]:
n, span = 20, 3.0
grid = np.linspace(-span, span, n)
canvas = np.zeros((n * 28, n * 28))
model.eval()
with torch.no_grad():
    for i, z2 in enumerate(grid):
        for j, z1 in enumerate(grid):
            z = torch.tensor([[z1, z2]], dtype=torch.float, device=device)
            img = model.decode(z).cpu().reshape(28, 28).numpy()
            canvas[(n - 1 - i) * 28:(n - i) * 28, j * 28:(j + 1) * 28] = img

plt.figure(figsize=(9, 9))
plt.imshow(canvas, cmap="gray", extent=[-span, span, -span, span])
plt.xlabel("z1"); plt.ylabel("z2"); plt.title("Latent manifold: digits decoded across the 2D latent space")
plt.show()

## Step 8 — Generate from pure noise

Sample `z ~ N(0, I)` and decode. These are brand-new digits the model invented — never in the training set.

In [ ]:
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM, device=device)
    gen = model.decode(z).cpu()

fig, ax = plt.subplots(8, 8, figsize=(8, 8))
for i, a in enumerate(ax.flat):
    a.imshow(gen[i, 0], cmap="gray"); a.axis("off")
plt.suptitle("Generated digits from N(0, I)"); plt.show()

## Deliverable checklist

| # | Deliverable | Where |
|---|---|---|
| 1 | Working convolutional VAE | `ConvVAE` (Step 2) |
| 2 | Training loss decreasing | loss curve (Step 4) |
| 3 | Reconstructions (sharper than linear) | Step 5 |
| 4 | Latent-space visualization | latent map (Step 6) + manifold grid (Step 7) |
| 5 | Generated samples from noise | Step 8 |

**What changed from Milestone 1:** only the layers (`Linear → Conv/ConvTranspose`) and the reconstruction loss
(`MSE → BCE`). The VAE skeleton — encode to a Gaussian, reparameterize, decode, train on reconstruction + KL —
is exactly the same. **Next (Weeks 5–6):** the same architecture scaled to 200K CelebA faces, then latent-space
interpolation and arithmetic on the trained face model.